In [32]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [3]:
spark = SparkSession.builder.appName("Airbnb London£").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/06 23:10:19 WARN Utils: Your hostname, kachi, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/06 23:10:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 23:10:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
listings = spark.read.csv(
    "listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE",
)
listings.show(5)

26/04/06 23:10:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+--------------------+--------------+------------+---------------+--------------------+--------------------+---------------------+--------------------+-------+--------------------+---------+----------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+--------------------+--------------------+------------------+-------------------+-------------------------+--------------------+--------------------+----------------------+--------------------+----------------------+----------------------------+--------+---------+--------------------+---------------+------------+---------+--------------+--------+----+--------------------+-------+--------------+--------------+----------------------+----------------------+----------------------+----------------------+----------------------+----------------------+----------------+----------------+---------------+---------------+---------------+----------------+---------------------+---

In [5]:
for prop in listings.schema:
    print(prop)

StructField('id', LongType(), True)
StructField('listing_url', StringType(), True)
StructField('scrape_id', LongType(), True)
StructField('last_scraped', DateType(), True)
StructField('source', StringType(), True)
StructField('name', StringType(), True)
StructField('description', StringType(), True)
StructField('neighborhood_overview', StringType(), True)
StructField('picture_url', StringType(), True)
StructField('host_id', IntegerType(), True)
StructField('host_url', StringType(), True)
StructField('host_name', StringType(), True)
StructField('host_since', DateType(), True)
StructField('host_location', StringType(), True)
StructField('host_about', StringType(), True)
StructField('host_response_time', StringType(), True)
StructField('host_response_rate', StringType(), True)
StructField('host_acceptance_rate', StringType(), True)
StructField('host_is_superhost', StringType(), True)
StructField('host_thumbnail_url', StringType(), True)
StructField('host_picture_url', StringType(), True)


In [6]:
neighbourhoods = listings.select(listings.neighbourhood_cleansed)
neighbourhoods.show(10, truncate=False)

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|Islington             |
|Kensington and Chelsea|
|Westminster           |
|Wandsworth            |
|Tower Hamlets         |
|Richmond upon Thames  |
|Haringey              |
|Hammersmith and Fulham|
|Hammersmith and Fulham|
|Southwark             |
+----------------------+
only showing top 10 rows


In [ ]:
high_score_listings = listings \
    .filter((listings.review_scores_location > 4.5) & (listings.number_of_reviews > 100)) \
    .filter(listings.price.isNotNull()) \
    .select("id", "name", "price", "review_scores_location", "number_of_reviews")

high_score_listings.show(20, truncate=False)

+--------+------------------------------------------------------------+---------+----------------------+-----------------+
|id      |name                                                        |price    |review_scores_location|number_of_reviews|
+--------+------------------------------------------------------------+---------+----------------------+-----------------+
|13499416|* by Wembley Stadium, 4 mins walk                           |$1,000.00|4.61                  |499              |
|37421626|New 2 bed apartment in Covent Garden                        |$1,000.00|4.86                  |148              |
|42597583|Bed in 4-Bed Female Dorm with Ensuite Bathroom              |$1,000.00|4.83                  |186              |
|9408750 |New 2Bed with Air Con in Kensington                         |$1,026.00|4.64                  |129              |
|14245764|4 Bed Twickenham House - Private Parking, Garden            |$1,277.00|4.85                  |265              |
|544824  |220+ r

In [57]:
from pyspark.sql.functions import regexp_replace

good_deals_df = high_score_listings.withColumn(
    "price_num", regexp_replace("price", "[$,]", "").cast("float")
) \
.filter(col('price_num') < 100) \
.sort(
    ["price_num", "review_scores_location", "number_of_reviews"],
    ascending=[True, False, False],
)

good_deals_df.schema["price_num"]

StructField('price_num', FloatType(), True)

In [58]:
good_deals_df.show(20, truncate=False)

+-------------------+-------------------------------------------------+------+----------------------+-----------------+---------+
|id                 |name                                             |price |review_scores_location|number_of_reviews|price_num|
+-------------------+-------------------------------------------------+------+----------------------+-----------------+---------+
|23443517           |Single Room, Garden View                         |$17.00|4.74                  |199              |17.0     |
|24978338           |Bed in an 18 Person Shared Dormitory             |$20.00|4.74                  |146              |20.0     |
|1146412423743825165|Bed in a 24 Person Shared Dorm                   |$20.00|4.64                  |287              |20.0     |
|34663126           |Bed in 12 Bed Dorm with Shared Bathroom          |$20.00|4.57                  |170              |20.0     |
|34662820           |Bed in 8 Bed Dorm                                |$21.00|4.65        

In [63]:
good_deals_df.count()

2122

In [66]:
good_deals_df.write.format("csv").save("good_deals.csv")